In [ ]:
dbutils.widgets.text("catalog", "")
catalog = dbutils.widgets.get("catalog")
if not catalog:
    raise ValueError("Set the catalog parameter before refreshing the taxi-zone lookup.")

source_path = f"/Volumes/{catalog}/landing/trips/reference/taxi_zone_lookup.csv"
zones = (
    spark.read.schema("LocationID INT, Borough STRING, Zone STRING, service_zone STRING")
    .option("header", True)
    .option("enforceSchema", False)
    .option("mode", "FAILFAST")
    .csv(source_path)
)
if zones.isEmpty():
    raise ValueError(f"The taxi-zone lookup is empty: {source_path}")

zones.createOrReplaceTempView("taxi_zones_input")
catalog_identifier = catalog.replace("`", "``")
spark.sql(
    """
    CREATE OR REPLACE TABLE IDENTIFIER(:target) USING DELTA
    AS SELECT LocationID, Borough, Zone, service_zone FROM taxi_zones_input
    """,
    args={"target": f"`{catalog_identifier}`.bronze.taxi_zones"},
)
